# 03 - Privacy Experiments

This notebook applies a simple and defensible de-identification protocol.

The implementation lives in `src/privacy/reporting.py`, so this notebook stays focused on:

- fixed privacy choices;
- optional training launch;
- original vs de-identified comparison tables;
- inline plots for interpretation.

Final report tables and figures are still regenerated with `scripts/reporting/make_plots.py`.


In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().resolve().parent
else:
    PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import src.configs as configs_module
import src.privacy.reporting as privacy_reporting

configs_module = importlib.reload(configs_module)
privacy_reporting = importlib.reload(privacy_reporting)

PYTHON_BIN = configs_module.resolve_python_bin()
TRAIN_SCRIPT = configs_module.TRAIN_SCRIPT
MAKE_PLOTS_SCRIPT = configs_module.MAKE_PLOTS_SCRIPT
DATA_ROOT = configs_module.DEFAULT_DATA_ROOT
RESULTS_SUMMARY_PATH = configs_module.RESULTS_FINAL_TABLES_DIR / "results_summary.csv"

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 240)

print("Project root:", PROJECT_ROOT)
print("Python:", PYTHON_BIN)
print("Training entrypoint:", TRAIN_SCRIPT)
print("Dataset root:", DATA_ROOT)


## 3.1 Protocol

The protocol is deliberately small:

1. Keep original baselines frozen.
2. Use fixed moderate values for `blur`, `mosaic` and `crop/context removal`.
3. Apply the same settings to `ResNet18`, `Swin-T` and `ViT-B/16`.
4. Compare accuracy, balanced accuracy, macro F1, precision, recall and loss.
5. Use ViT attention analysis separately in `04_AttentionMaps.ipynb`.


In [ ]:
RUN_FIXED_DEID = False
SKIP_COMPLETED = True
SAVE_COMPARISON_CSV = True

fixed_filter_df = privacy_reporting.fixed_filter_table()
display(fixed_filter_df[["label", "mode", "intensity", "justification"]])


## 3.2 Original Baselines

These are the frozen original-image models used as reference points for performance drops.


In [ ]:
baseline_df, missing_baselines = privacy_reporting.load_original_baselines()

if missing_baselines:
    print("Missing original baselines:")
    for run_name in missing_baselines:
        print("-", run_name)

if baseline_df.empty:
    print("No original baseline metrics loaded.")
else:
    display(
        baseline_df[
            [
                "model",
                "test_accuracy",
                "test_balanced_accuracy",
                "test_f1_macro",
                "test_precision_macro",
                "test_recall_macro",
                "test_loss",
                "best_epoch",
                "run_name",
            ]
        ].style.format(
            {
                "test_accuracy": "{:.4f}",
                "test_balanced_accuracy": "{:.4f}",
                "test_f1_macro": "{:.4f}",
                "test_precision_macro": "{:.4f}",
                "test_recall_macro": "{:.4f}",
                "test_loss": "{:.4f}",
            }
        )
    )


## 3.3 Visual Check Of Fixed Filters

This preview checks that the chosen values are moderate: visible privacy transformation, but not total destruction of the face.


In [ ]:
fig, sample_image_path = privacy_reporting.plot_fixed_filter_preview(DATA_ROOT)
plt.show()
print("Preview image:", sample_image_path)


## 3.4 Planned Fixed De-id Runs

Training is disabled by default. Set `RUN_FIXED_DEID = True` only when you intentionally want to launch missing runs.


In [ ]:
deid_configs = privacy_reporting.get_deid_configs()
planned_df = privacy_reporting.build_config_table(deid_configs)
display(planned_df)

commands_df = privacy_reporting.build_command_table(
    configs=deid_configs,
    train_script=TRAIN_SCRIPT,
    python_bin=PYTHON_BIN,
    data_root=DATA_ROOT,
    skip_completed=SKIP_COMPLETED,
)
display(commands_df)


In [ ]:
if not RUN_FIXED_DEID:
    print("Fixed de-id training disabled. Set RUN_FIXED_DEID = True to launch missing runs.")
else:
    privacy_reporting.run_configs(
        configs=deid_configs,
        stage_name="fixed de-id",
        train_script=TRAIN_SCRIPT,
        python_bin=PYTHON_BIN,
        data_root=DATA_ROOT,
        skip_completed=SKIP_COMPLETED,
    )


## 3.5 Overall Impact

This compares original vs fixed de-identification runs. The CSV is kept because the reporting and validation scripts use it.


In [ ]:
deid_df, missing_deid = privacy_reporting.load_fixed_deid_metrics()

if missing_deid:
    print("Missing fixed de-id metrics:")
    for run_name in missing_deid:
        print("-", run_name)

fixed_comparison_df = privacy_reporting.build_fixed_deid_comparison(
    baseline_df=baseline_df,
    deid_df=deid_df,
)

if fixed_comparison_df.empty:
    print("No fixed de-id comparison available yet.")
else:
    if SAVE_COMPARISON_CSV:
        output_path = privacy_reporting.save_fixed_deid_comparison(fixed_comparison_df)
        print("Saved fixed de-id comparison to:", output_path)

    display(
        fixed_comparison_df[
            [
                "model",
                "condition",
                "privacy_intensity",
                "test_accuracy",
                "test_balanced_accuracy",
                "test_f1_macro",
                "delta_f1_macro",
                "test_precision_macro",
                "test_recall_macro",
                "test_loss",
                "best_epoch",
                "run_name",
            ]
        ].style.format(
            {
                "privacy_intensity": "{:.2f}",
                "test_accuracy": "{:.4f}",
                "test_balanced_accuracy": "{:.4f}",
                "test_f1_macro": "{:.4f}",
                "delta_f1_macro": "{:+.4f}",
                "test_precision_macro": "{:.4f}",
                "test_recall_macro": "{:.4f}",
                "test_loss": "{:.4f}",
            }
        )
    )


## 3.6 Plots

Plots are shown inline. Final saved report figures are created with `scripts/reporting/make_plots.py`.


In [ ]:
if "fixed_comparison_df" not in globals() or fixed_comparison_df.empty:
    print("No fixed de-id comparison available to plot.")
else:
    fig = privacy_reporting.plot_fixed_deid_impact(fixed_comparison_df, baseline_df)
    plt.show()


In [ ]:
if "fixed_comparison_df" not in globals() or fixed_comparison_df.empty:
    print("No fixed de-id comparison available to plot.")
else:
    fig = privacy_reporting.plot_macro_f1_drop(fixed_comparison_df)
    plt.show()


## 3.7 Official Reporting Output

The final consolidated table is `results/tables/final/results_summary.csv`, recreated with:

```powershell
.\.venv\Scripts\python.exe .\scripts\reporting\make_plots.py
```


In [ ]:
try:
    results_summary_df = privacy_reporting.load_results_summary(RESULTS_SUMMARY_PATH)
    privacy_rows = results_summary_df[
        results_summary_df["transformation"].isin(
            ["crop_context_removal", "blur", "mosaic", "canny", "noise"]
        )
    ].copy()

    display(
        privacy_rows[
            [
                "model",
                "transformation",
                "privacy_intensity",
                "accuracy",
                "precision",
                "recall",
                "macro_f1",
                "macro_f1_drop_vs_clean",
                "summary_source",
            ]
        ].style.format(
            {
                "privacy_intensity": "{:.2f}",
                "accuracy": "{:.4f}",
                "precision": "{:.4f}",
                "recall": "{:.4f}",
                "macro_f1": "{:.4f}",
                "macro_f1_drop_vs_clean": "{:+.4f}",
            }
        )
    )
    print("Official summary table:", RESULTS_SUMMARY_PATH)
except FileNotFoundError as error:
    print(error)
    print(f"Run: {PYTHON_BIN} {MAKE_PLOTS_SCRIPT}")


## 3.8 Next Step

The simple de-identification experiment is closed here. ViT attention maps, landmark regions and inverse-attention masking are handled in `04_AttentionMaps.ipynb`.
